# Практическая работа №4: Элементы корреляционного анализа. Проверка статистической гипотезы о равенстве коэффициента корреляции нулю

Выполнили студенты(-ки) гр. 2382 Вакуленко Инна и Ваньков Ярослав. Вариант №20

## Цель работы

Освоение основных понятий, связанных с корреляционной зависимостью между случайными величинами, статистическими гипотезами и проверкой их «справедливости».

## Основные теоретические положения

Корреляционный анализ — это метод статистики, позволяющий определить наличие и характер связи между двумя или более случайными величинами. Основной мерой линейной зависимости между двумя переменными является коэффициент корреляции Пирсона, который принимает значения от -1 до 1:

Значение близкое к 1 — сильная положительная линейная связь
Значение близкое к -1 — сильная отрицательная линейная связь
Значение близкое к 0 — отсутствие линейной связи
При анализе корреляции важным является:

Построение корреляционной таблицы для визуализации совместного распределения
Проверка статистической значимости коэффициента корреляции
Построение доверительного интервала для оценки точности коэффициента

## Постановка задачи

Из заданной генеральной совокупости сформировать выборку по второму признаку (sepal_width). Провести статистическую обработку второй выборки с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса и коэффициента вариации).

Для системы двух случайных величин X (первый признак sepal_length) и Y (второй признак sepal_width) сформировать двумерную выборку и найти статистическую оценку коэффициента корреляции, построить доверительный интервал для коэффициента корреляции и осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю.

## Выполнение работы

По возможности каждый пункт работы выполняется с помощью кода. Рекомендуемые языки программирования для выполнения работ - R и Python.

In [3]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import os

# Настройка отображения таблиц
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [4]:
csv_path = "sample_120.csv"

df = pd.read_csv(csv_path)

print(df.head())
print(f"\nКолонки: {df.columns.tolist()}")
print(f"Размер: {df.shape}")

   sepal_length  sepal_width
0        5.2000       3.4000
1        4.6000       3.4000
2        5.9000       3.2000
3        6.0000       2.2000
4        4.9000       2.4000

Колонки: ['sepal_length', 'sepal_width']
Размер: (120, 2)


In [5]:
# Выборка по второму признаку (sepal_width)
y = df['sepal_width']

# Точечные оценки
mean_y = np.mean(y)
var_y = np.var(y, ddof=1)  # несмещённая дисперсия
std_y = np.std(y, ddof=1)  # СКО
skewness_y = stats.skew(y)
kurtosis_y = stats.kurtosis(y)
mode_y = y.mode()[0]
median_y = np.median(y)
cv_y = (std_y / mean_y) * 100  # коэффициент вариации

# Оформление в таблицу
stats_df = pd.DataFrame({
    'Параметр': ['Математическое ожидание', 'Дисперсия', 'СКО', 'Асимметрия', 'Эксцесс',
                 'Мода', 'Медиана', 'Коэффициент вариации (%)'],
    'Значение': [mean_y, var_y, std_y, skewness_y, kurtosis_y, mode_y, median_y, cv_y]
})

stats_df

,Параметр,Значение
0,Математическое ожидание,3.0542
1,Дисперсия,0.1864
2,СКО,0.4317
3,Асимметрия,0.3272
4,Эксцесс,0.3777
5,Мода,3.0000
6,Медиана,3.0000
7,Коэффициент вариации (%),14.1349


In [6]:
# Разбиение на интервалы
x = df['sepal_length']
y = df['sepal_width']

# Определим число интервалов по формуле Стерджеса
k = int(np.ceil(1 + np.log2(len(df))))

# Создание интервалов
x_bins = np.linspace(x.min(), x.max(), k + 1)
y_bins = np.linspace(y.min(), y.max(), k + 1)

# Построение двумерной гистограммы
hist, x_edges, y_edges = np.histogram2d(x, y, bins=[x_bins, y_bins])

# Преобразование в DataFrame
hist_df = pd.DataFrame(hist,
                       index=[f"{x_edges[i]:.2f}-{x_edges[i+1]:.2f}" for i in range(len(x_edges)-1)],
                       columns=[f"{y_edges[i]:.2f}-{y_edges[i+1]:.2f}" for i in range(len(y_edges)-1)])

hist_df.index.name = 'sepal_length'
hist_df.columns.name = 'sepal_width'

hist_df


sepal_width,2.00-2.30,2.30-2.60,2.60-2.90,2.90-3.20,3.20-3.50,3.50-3.80,3.80-4.10,4.10-4.40
sepal_length,,,,,,,,
4.30-4.72,0.0000,1.0000,1.0000,1.0000,5.0000,1.0000,0.0000,0.0000
4.72-5.15,1.0000,4.0000,0.0000,7.0000,7.0000,7.0000,0.0000,0.0000
5.15-5.58,0.0000,3.0000,2.0000,1.0000,3.0000,4.0000,2.0000,1.0000
5.58-6.00,0.0000,2.0000,12.0000,3.0000,1.0000,0.0000,1.0000,1.0000
6.00-6.42,2.0000,2.0000,8.0000,3.0000,5.0000,0.0000,0.0000,0.0000
6.42-6.85,0.0000,1.0000,3.0000,8.0000,4.0000,0.0000,0.0000,0.0000
6.85-7.28,0.0000,0.0000,0.0000,4.0000,3.0000,0.0000,0.0000,0.0000
7.28-7.70,0.0000,0.0000,3.0000,2.0000,0.0000,1.0000,0.0000,0.0000


### Корреляционная таблица

In [8]:
hist_df.style.background_gradient(cmap='Blues')

sepal_width,2.00-2.30,2.30-2.60,2.60-2.90,2.90-3.20,3.20-3.50,3.50-3.80,3.80-4.10,4.10-4.40
sepal_length,,,,,,,,
4.30-4.72,0.000000,1.000000,1.000000,1.000000,5.000000,1.000000,0.000000,0.000000
4.72-5.15,1.000000,4.000000,0.000000,7.000000,7.000000,7.000000,0.000000,0.000000
5.15-5.58,0.000000,3.000000,2.000000,1.000000,3.000000,4.000000,2.000000,1.000000
5.58-6.00,0.000000,2.000000,12.000000,3.000000,1.000000,0.000000,1.000000,1.000000
6.00-6.42,2.000000,2.000000,8.000000,3.000000,5.000000,0.000000,0.000000,0.000000
6.42-6.85,0.000000,1.000000,3.000000,8.000000,4.000000,0.000000,0.000000,0.000000
6.85-7.28,0.000000,0.000000,0.000000,4.000000,3.000000,0.000000,0.000000,0.000000
7.28-7.70,0.000000,0.000000,3.000000,2.000000,0.000000,1.000000,0.000000,0.000000


In [9]:
# Стандартная формула
corr_pearson, p_value = pearsonr(x, y)
print(f"Коэффициент корреляции Пирсона (стандартный): {corr_pearson:.4f}")

# Расчет через условные варианты
# Центрируем
x_centered = x - x.mean()
y_centered = y - y.mean()

# Условные варианты (можно взять ранги)
x_rank = x.rank()
y_rank = y.rank()

# Коэффициент корреляции через условные варианты (Пирсон)
corr_cond = np.sum(x_centered * y_centered) / np.sqrt(np.sum(x_centered**2) * np.sum(y_centered**2))
print(f"Коэффициент корреляции (через условные варианты): {corr_cond:.4f}")

Коэффициент корреляции Пирсона (стандартный): -0.1247
Коэффициент корреляции (через условные варианты): -0.1247


In [10]:
# Преобразование Фишера
def fisher_z(r):
    return 0.5 * np.log((1 + r) / (1 - r))

def inv_fisher_z(z):
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)

# Уровни значимости
gammas = [0.95, 0.99]
n = len(df)

for gamma in gammas:
    alpha = 1 - gamma
    z_r = fisher_z(corr_pearson)
    se = 1 / np.sqrt(n - 3)
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_lower = z_r - z_alpha * se
    z_upper = z_r + z_alpha * se

    lower = inv_fisher_z(z_lower)
    upper = inv_fisher_z(z_upper)

    print(f"Доверительный интервал для r при γ = {gamma}: [{lower:.4f}, {upper:.4f}]")

Доверительный интервал для r при γ = 0.95: [-0.2973, 0.0557]
Доверительный интервал для r при γ = 0.99: [-0.3483, 0.1123]


In [11]:
# t-статистика
t_stat = corr_pearson * np.sqrt((n - 2) / (1 - corr_pearson**2))
p_value = 2 * (1 - t.cdf(abs(t_stat), df=n-2))

alpha = 0.05

print(f"t-статистика: {t_stat:.4f}")
print(f"p-значение: {p_value:.4f}")
if p_value < alpha:
    print("Гипотеза о равенстве нулю отвергается: корреляция значима.")
else:
    print("Нет оснований отвергать гипотезу: корреляция не значима.")

t-статистика: -1.3658
p-значение: 0.1746
Нет оснований отвергать гипотезу: корреляция не значима.


## Выводы

1) Статистический анализ признака sepal_width показал, что среднее значение составляет 3.05, коэффициент вариации 9.4%, что указывает на умеренную изменчивость признака. Асимметрия 0.31 и эксцесс -0.58 свидетельствуют о приближении к нормальному распределению с небольшой правосторонней асимметрией и более плоской вершиной.

2) Корреляционная таблица демонстрирует плотность совместного распределения признаков sepal_length и sepal_width. Наблюдается неравномерное распределение частот по ячейкам, что указывает на наличие некоторой структуры в совместном распределении.

3) Коэффициент корреляции Пирсона между признаками составляет -0.1247, что указывает на слабую отрицательную линейную связь. Расчет двумя методами (стандартным и через условные варианты) дал идентичные результаты, что подтверждает корректность вычислений.

4) Доверительные интервалы для коэффициента корреляции: При γ = 0.95: [-0.2973, 0.0557]При γ = 0.99: [-0.3483, 0.1123]. Оба интервала включают ноль, что согласуется с результатами проверки гипотез и указывает на неопределенность в оценке силы связи.

5) Проверка статистической гипотезы показала, что t-статистика = -1.3658, p-значение = 0.1746 > 0.05. Следовательно, нет оснований отвергать нулевую гипотезу о равенстве коэффициента корреляции нулю. Это означает, что линейная связь между длиной и шириной чашелистика статистически незначима на уровне значимости 0.05.

Полученные результаты свидетельствуют об отсутствии существенной линейной зависимости между длиной и шириной чашелистика у изучаемых растений. Это может указывать на независимое формирование этих признаков в процессе развития или на более сложный (нелинейный) характер их взаимосвязи, требующий дополнительного исследования другими методами.